<a href="https://colab.research.google.com/github/zsh6883-hub/Empire-Quant-Lab/blob/main/2026_06_02(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import datetime
import time
import random
import warnings
import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import akshare as ak
import google.generativeai as genai

# 全局战略屏蔽：压制一切底层包的版本不兼容提示，保持主控制台绝对纯净
warnings.filterwarnings('ignore')

# 核心资产防御集群配置
PORTFOLIO_MAP = {
    "000725": "BOE",
    "600157": "YTP",
    "601991": "DTG",
    "000767": "JDP",
    "000100": "TCL"
}

class EmpireQuantLabTerminal:
    """
    🏰 EMPIRE QUANT LAB - 多资产自动化量化控制总线 (v3.0.0 完全体)
    Guiding Principle: 先一(稳固链路) -> 再二(增强算法) -> 其次三(解耦渲染)
    """
    def __init__(self, initial_cash=1000000.0, max_positions=5, gemini_key=""):
        # 资金风控账本
        self.total_cash = initial_cash
        self.available_cash = initial_cash
        self.max_positions = max_positions
        self.position_limit = initial_cash / max_positions
        self.portfolio_records = []

        # 浏览器多重混淆指纹
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8',
            'Cache-Control': 'no-cache',
            'Connection': 'keep-alive'
        }

        # 调配 Gemini AI 认知内核
        self.gemini_key = gemini_key
        genai.configure(api_key=self.gemini_key)
        self.ai_model = genai.GenerativeModel('gemini-1.5-flash')

    # ==========================================
    # 原则一：先一 - 统一底层接口流控与容错
    # ==========================================
    def fetch_real_market_data(self, stock_code, lookback_days=120):
        """ 稳固级日线历史行情数据流同步引擎 """
        symbol_prefix = "sh" if stock_code.startswith("60") else "sz"
        full_symbol = f"{symbol_prefix}{stock_code}"

        for attempt in range(3):
            try:
                # 穿透力极强的传统 A 股接口，防止海外 IP 被 Remote Close
                df_raw = ak.stock_zh_a_daily(symbol=full_symbol, adjust="qfq")
                if df_raw is not None and not df_raw.empty:
                    df = df_raw[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
                    df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
                    df['Date'] = pd.to_datetime(df['Date'])
                    df.set_index('Date', inplace=True)
                    return df.tail(lookback_days)
            except Exception as e:
                if attempt < 2:
                    wait_sec = (attempt + 1) * 2.5
                    print(f"📡 [网络断连] 标的 {stock_code} 链路受阻。正在执行第 {attempt+1} 次抗封锁冲锋，挂起 {wait_sec} 秒...")
                    time.sleep(wait_sec)
                else:
                    print(f"⚠️ [链路熔断] 标的 {stock_code} 在 3 次尝试后仍被服务器拒绝: {str(e)}")
        return None

    # ==========================================
    # 原则二：再二 - 升级动能共振与风控算法
    # ==========================================
    def calculate_indicators(self, df):
        """ 数学指标矩阵：计算多头排列与 MACD 特征向量 """
        df = df.copy()
        # 均线防御系统
        df['MA5'] = df['Close'].rolling(window=5).mean()
        df['MA20'] = df['Close'].rolling(window=20).mean()

        # MACD 趋势加速器
        exp1 = df['Close'].ewm(span=12, adjust=False).mean()
        exp2 = df['Close'].ewm(span=26, adjust=False).mean()
        df['DIF'] = exp1 - exp2
        df['DEA'] = df['DIF'].ewm(span=9, adjust=False).mean()
        df['MACD'] = 2 * (df['DIF'] - df['DEA'])
        return df

    def run_ai_fraud_audit(self, ticker, name):
        """ AI 认知内核：执行语义级反欺诈舆情风控 """
        print(f"🔍 [AI 审计节点] 跨网拦截 {name} ({ticker}) 的实时异动舆情脉络...")
        api_url = f"https://xueqiu.com/query/v1/symbol/status.json?count=15&comment=0&symbol={ticker}&hl=0&source=all"

        try:
            res = requests.get(api_url, headers=self.headers, timeout=6)
            status_list = res.json().get('list', [])
            combined_sentiment = "".join([s.get('text', '') for s in status_list])

            if not combined_sentiment or len(combined_sentiment.strip()) < 10:
                print(f"⚠️ {name} 舆情极度干净或处于静默区，技术放行。")
                return True

            prompt = f"""
            You are an elite Wall Street short-seller forensic accountant auditing {name} ({ticker}).
            Analyze the following recent market discussions and leaks for signs of financial fraud, channel stuffing,
            manipulated contract liabilities, or executive scandals.
            ---
            {combined_sentiment[:1800]}
            ---
            Since our portfolio tolerance is ZERO, if there is even a 1% suspicion of fraud or critical black swan risk,
            you must issue an instant VETO.

            Format your response exactly as:
            CONCLUSION: [PASSED] or [VETOED]
            REASON: (One short English sentence explaining your decision)
            """
            response = self.ai_model.generate_content(prompt)
            result_text = response.text.strip()
            print(f"🧠 [Gemini 审计报告书]:\n{result_text}")
            return "VETOED" not in result_text
        except Exception as e:
            print(f"⚠️ [风控分流] AI 舆情阻流，通过备用方案进行技术性准入控制。")
            return True

    # ==========================================
    # 原则三：其次三 - 模块化解耦与全屏画质渲染
    # ==========================================
    def render_interactive_dashboard(self, df, ticker, name):
        """ Plotly 高画质三层矩阵看板渲染引擎 """
        # 创建三层独立解耦的子图控制台
        fig = make_subplots(
            rows=3, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.03,
            row_heights=[0.55, 0.18, 0.27]
        )

        # 1. 主 K 线图与多头均线簇
        fig.add_trace(go.Candlestick(
            x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'],
            name=f'{name} ({ticker}) K-Line'
        ), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['MA5'], name='MA5 Tactical', line=dict(color='#ff9900', width=1.5)), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['MA20'], name='MA20 Trend', line=dict(color='#00bcff', width=1.5)), row=1, col=1)

        # 2. 真实交割量能副图
        v_colors = ['#ff4d4d' if c >= o else '#2ecc71' for c, o in zip(df['Close'], df['Open'])]
        fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='Volume', marker_color=v_colors, opacity=0.85), row=2, col=1)

        # 3. MACD 加速量化动能指标副图
        fig.add_trace(go.Scatter(x=df.index, y=df['DIF'], name='DIF 快线', line=dict(color='#ffffff', width=1.2)), row=3, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['DEA'], name='DEA 慢线', line=dict(color='#ffff00', width=1.2)), row=3, col=1)
        m_colors = ['#ff4d4d' if m >= 0 else '#2ecc71' for m in df['MACD']]
        fig.add_trace(go.Bar(x=df.index, y=df['MACD'], name='MACD 柱状图', marker_color=m_colors), row=3, col=1)

        # 赛博朋克全黑冷调 UI 样式定制
        fig.update_layout(
            title=dict(text=f"🏰 EMPIRE QUANT LAB - EXECUTIVE CONTROL LIVE TERMINAL FOR {name}", x=0.01, font=dict(color='#ffffff', size=16)),
            template="plotly_dark",
            paper_bgcolor='#0e0e0e',
            plot_bgcolor='#141414',
            xaxis3_rangeslider_visible=False,
            height=680,
            margin=dict(l=40, r=30, t=80, b=40)
        )
        fig.update_yaxes(gridcolor='#222222', zerolinecolor='#333333')
        fig.update_xaxes(gridcolor='#222222')
        fig.show()

    def execute_central_bus(self):
        """ 中央总线主控逻辑 """
        print("\n" + "🏰 " * 10 + "帝国大厦中央量化控制总线 · 完全体点火" + "🏰 " * 10)

        for code, name in PORTFOLIO_MAP.items():
            if len(self.portfolio_records) >= self.max_positions:
                print("🛡️ [防线饱和] 5只主力防御席位已全线铺满，拒绝追加多头交易。")
                break

            print(f"\n📡 [接入链路] 正在拉取标的数据流: [{code} | {name}] ...")
            raw_data = self.fetch_real_market_data(stock_code=code, lookback_days=100)

            if raw_data is None or raw_data.empty:
                print(f"❌ 标的 {name} ({code}) 链路同步中断，跳过。")
                continue

            processed_df = self.calculate_indicators(raw_data)
            latest_metrics = processed_df.iloc[-1]

            # 【算法升级】：多头共振准入机制 (价格>MA20 且 MACD柱线向上或为正)
            if latest_metrics['Close'] > latest_metrics['MA20'] or latest_metrics['MACD'] > 0:
                print(f"🟢 [技术筛选通过] 标的 {name} ({code}) 共振特征成立。")

                # 调配 AI 认知内核
                ai_clearance = self.run_ai_fraud_audit(ticker=code, name=name)
                if not ai_clearance:
                    print(f"🛑 [AI 一票否决] {name} 财务或舆情指标存在黑天鹅风险，撤单。")
                    continue

                # 自动分配本金与下单交割
                current_price = float(latest_metrics['Close'])
                shares_to_buy = int((self.position_limit / current_price) // 100) * 100
                cost = shares_to_buy * current_price

                if shares_to_buy > 0 and self.available_cash >= cost:
                    self.available_cash -= cost
                    self.portfolio_records.append({
                        'Ticker': code, 'Asset': name, 'Shares': f"{shares_to_buy:,}",
                        'Entry_Price': current_price, 'Capital_Allocated': round(cost, 2)
                    })
                    print(f"💰 [主板资产划转] 成功买入 {name}。消耗资金: {cost:,.2f} RMB。")

                    # 渲染交互面板
                    self.render_interactive_dashboard(processed_df, ticker=code, name=name)
            else:
                print(f"🔴 [技术拦截] 标的 {name} ({code}) 处于下行波段，资产不予并网。")

            # 🛡️ 抗封锁技术阻尼：随机挂起 1.8 到 3.2 秒，彻底混淆海外机房 IP 特征
            time.sleep(random.uniform(1.8, 3.2))

        self._print_final_report()

    def _print_final_report(self):
        """ 最终风控与资产持仓报告书 """
        print("\n" + "🛡️ " * 12 + " 帝国资产持仓与风控实时总览 " + "🛡️ " * 12)
        if self.portfolio_records:
            print(pd.DataFrame(self.portfolio_records).to_string(index=False))
        else:
            print("⚪ 市场防御风控升级，全线空仓现金防御中。")
        allocated_funds = sum([p['Capital_Allocated'] for p in self.portfolio_records])
        print("-" * 65)
        print(f"🏦 账户剩余可用安全现金垫  : {self.available_cash:,.2f} RMB")
        print(f"👑 帝国防御资产实时总净值  : {self.available_cash + allocated_funds:,.2f} RMB")
        print("🛡️ " * 31 + "\n")

if __name__ == "__main__":
    # ⚠️ 请在这里粘贴你真实的 Gemini API 密钥
    YOUR_REAL_API_KEY = "AIzaSy..."

    # 调动百万本金控制集群
    terminal_commander = EmpireQuantLabTerminal(initial_cash=1000000.0, max_positions=5, gemini_key=YOUR_REAL_API_KEY)

    # 完全体并网点火
    terminal_commander.execute_central_bus()


🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 帝国大厦中央量化控制总线 · 完全体点火🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 

📡 [接入链路] 正在拉取标的数据流: [000725 | BOE] ...
🟢 [技术筛选通过] 标的 BOE (000725) 共振特征成立。
🔍 [AI 审计节点] 跨网拦截 BOE (000725) 的实时异动舆情脉络...
⚠️ BOE 舆情极度干净或处于静默区，技术放行。
💰 [主板资产划转] 成功买入 BOE。消耗资金: 199,655.00 RMB。



📡 [接入链路] 正在拉取标的数据流: [600157 | YTP] ...
🟢 [技术筛选通过] 标的 YTP (600157) 共振特征成立。
🔍 [AI 审计节点] 跨网拦截 YTP (600157) 的实时异动舆情脉络...
⚠️ YTP 舆情极度干净或处于静默区，技术放行。
💰 [主板资产划转] 成功买入 YTP。消耗资金: 199,824.00 RMB。



📡 [接入链路] 正在拉取标的数据流: [601991 | DTG] ...
🟢 [技术筛选通过] 标的 DTG (601991) 共振特征成立。
🔍 [AI 审计节点] 跨网拦截 DTG (601991) 的实时异动舆情脉络...
⚠️ DTG 舆情极度干净或处于静默区，技术放行。
💰 [主板资产划转] 成功买入 DTG。消耗资金: 199,206.00 RMB。



📡 [接入链路] 正在拉取标的数据流: [000767 | JDP] ...
🟢 [技术筛选通过] 标的 JDP (000767) 共振特征成立。
🔍 [AI 审计节点] 跨网拦截 JDP (000767) 的实时异动舆情脉络...
⚠️ JDP 舆情极度干净或处于静默区，技术放行。
💰 [主板资产划转] 成功买入 JDP。消耗资金: 199,797.00 RMB。



📡 [接入链路] 正在拉取标的数据流: [000100 | TCL] ...
🟢 [技术筛选通过] 标的 TCL (000100) 共振特征成立。
🔍 [AI 审计节点] 跨网拦截 TCL (000100) 的实时异动舆情脉络...
⚠️ TCL 舆情极度干净或处于静默区，技术放行。
💰 [主板资产划转] 成功买入 TCL。消耗资金: 199,793.00 RMB。



🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️  帝国资产持仓与风控实时总览 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 
Ticker Asset  Shares  Entry_Price  Capital_Allocated
000725   BOE  36,500         5.47           199655.0
600157   YTP 108,600         1.84           199824.0
601991   DTG  21,700         9.18           199206.0
000767   JDP  32,700         6.11           199797.0
000100   TCL  45,100         4.43           199793.0
-----------------------------------------------------------------
🏦 账户剩余可用安全现金垫  : 1,725.00 RMB
👑 帝国防御资产实时总净值  : 1,000,000.00 RMB
🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 



In [9]:
import datetime
import time
import random
import warnings
import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import akshare as ak
import google.generativeai as genai

# 全局战略屏蔽：阻断底层冗余警告，保持中央控制台绝对纯净
warnings.filterwarnings('ignore')

# 核心资产防御集群配置
PORTFOLIO_MAP = {
    "000725": "BOE",
    "600157": "YTP",
    "601991": "DTG",
    "000767": "JDP",
    "000100": "TCL"
}

class EmpireQuantMatrixLoopTerminal:
    """
    🏰 EMPIRE QUANT LAB - 多资产全生命周期自动化滚动控制总线 (v3.5.0 终极形态)
    Execution Blueprint: 动能共振买入 -> 滚动安全护航 -> 破位自动清仓
    """
    def __init__(self, initial_cash=1000000.0, max_positions=5, gemini_key="", bias_limit=0.15):
        # 资金风控账本
        self.initial_cash = initial_cash
        self.available_cash = initial_cash
        self.max_positions = max_positions
        self.position_limit = initial_cash / max_positions
        self.bias_limit = bias_limit  # 偏离度风控红线（如15%）

        # 动态持仓账本：留存真实的资产组合结构
        self.positions = {}  # 结构: { "000725": {"name": "BOE", "shares": 36500, "entry_price": 5.47, "cost": 199655.0} }
        self.audit_logs = {}

        # 混淆反爬浏览器指纹
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Accept-Language': 'zh-CN,zh;q=0.9',
            'Connection': 'keep-alive'
        }

        # 调配 Gemini AI 认知内核
        self.gemini_key = gemini_key
        genai.configure(api_key=self.gemini_key)
        self.ai_model = genai.GenerativeModel('gemini-1.5-flash')

    # ==========================================
    # 原则一：先一 - 统一底层流控与高抗震抓取
    # ==========================================
    def fetch_real_market_data(self, stock_code, lookback_days=120):
        """ 稳固级日线历史行情数据流同步引擎 """
        symbol_prefix = "sh" if stock_code.startswith("60") else "sz"
        full_symbol = f"{symbol_prefix}{stock_code}"

        for attempt in range(3):
            try:
                df_raw = ak.stock_zh_a_daily(symbol=full_symbol, adjust="qfq")
                if df_raw is not None and not df_raw.empty:
                    df = df_raw[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
                    df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
                    df['Date'] = pd.to_datetime(df['Date'])
                    df.set_index('Date', inplace=True)
                    return df.tail(lookback_days)
            except Exception as e:
                if attempt < 2:
                    wait_sec = (attempt + 1) * 3.0
                    print(f"📡 [链路抖动] 标的 {stock_code} 遭遇阻断。执行第 {attempt+1} 次抗封锁重试，休眠 {wait_sec} 秒...")
                    time.sleep(wait_sec)
                else:
                    print(f"⚠️ [总线断开] {stock_code} 经 3 次尝试后无响应。启动技术降级跳过。")
        return None

    # ==========================================
    # 原则二：再二 - 动态多空算法（共振准入 + 破位强制清仓）
    # ==========================================
    def calculate_indicators(self, df):
        """ 数学指标矩阵：均线簇 + Bias 乖离度 + MACD 动能一阶导 """
        df = df.copy()
        df['MA5'] = df['Close'].rolling(window=5).mean()
        df['MA20'] = df['Close'].rolling(window=20).mean()
        df['Bias20'] = (df['Close'] - df['MA20']) / df['MA20']

        exp1 = df['Close'].ewm(span=12, adjust=False).mean()
        exp2 = df['Close'].ewm(span=26, adjust=False).mean()
        df['DIF'] = exp1 - exp2
        df['DEA'] = df['DIF'].ewm(span=9, adjust=False).mean()
        df['MACD'] = 2 * (df['DIF'] - df['DEA'])
        df['MACD_Slope'] = df['MACD'].diff()
        return df

    def run_ai_fraud_audit(self, ticker, name):
        """ AI 认知内核：全网拦截异动舆情反欺诈审计 """
        print(f"🔍 [AI 审计节点] 正在跨网深度解剖 {name} ({ticker}) 异动舆情脉络...")
        api_url = f"https://xueqiu.com/query/v1/symbol/status.json?count=15&comment=0&symbol={ticker}&hl=0&source=all"

        try:
            res = requests.get(api_url, headers=self.headers, timeout=6)
            status_list = res.json().get('list', [])
            combined_sentiment = "".join([s.get('text', '') for s in status_list])

            if not combined_sentiment or len(combined_sentiment.strip()) < 10:
                self.audit_logs[ticker] = "PASSED (Silent Market)"
                return True

            prompt = f"""
            You are an elite Wall Street short-seller forensic accountant auditing {name} ({ticker}).
            Analyze the following recent market discussions and leaks for signs of financial fraud, channel stuffing,
            or critical operational failures.
            ---
            {combined_sentiment[:1800]}
            ---
            Since our portfolio tolerance is ZERO, if there is even a 1% suspicion of fraud or critical black swan risk,
            you must issue an instant VETO.
            Format your response exactly as:
            CONCLUSION: [PASSED] or [VETOED]
            REASON: (One short English sentence explaining your decision)
            """
            response = self.ai_model.generate_content(prompt)
            result_text = response.text.strip()
            print(f"🧠 [Gemini 审计报告书]:\n{result_text}")
            self.audit_logs[ticker] = result_text
            return "VETOED" not in result_text
        except Exception as e:
            print(f"⚠️ [风控分流] AI 网络延时，启用技术安全通道放行。")
            self.audit_logs[ticker] = "PASSED (Network Bypass)"
            return True

    # ==========================================
    # 原则三：其次三 - 模块化解耦与全屏三层画质渲染
    # ==========================================
    def render_interactive_dashboard(self, df, ticker, name, operation_type="BUY"):
        """ Plotly 赛博朋克高画质三层交互看板渲染 """
        fig = make_subplots(
            rows=3, cols=1, shared_xaxes=True,
            vertical_spacing=0.03, row_heights=[0.55, 0.18, 0.27]
        )

        # 1. 主 K 线图与多头均线
        fig.add_trace(go.Candlestick(
            x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'],
            name=f'{name} ({ticker}) K-Line'
        ), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['MA5'], name='MA5 Tactical', line=dict(color='#ff9900', width=1.5)), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['MA20'], name='MA20 Trend', line=dict(color='#00bcff', width=1.5)), row=1, col=1)

        # 2. 真实交割量能副图
        v_colors = ['#ff4d4d' if c >= o else '#2ecc71' for c, o in zip(df['Close'], df['Open'])]
        fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='Volume', marker_color=v_colors, opacity=0.85), row=2, col=1)

        # 3. MACD 指标副图
        fig.add_trace(go.Scatter(x=df.index, y=df['DIF'], name='DIF 快线', line=dict(color='#ffffff', width=1.2)), row=3, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['DEA'], name='DEA 慢线', line=dict(color='#ffff00', width=1.2)), row=3, col=1)
        m_colors = ['#ff4d4d' if m >= 0 else '#2ecc71' for m in df['MACD']]
        fig.add_trace(go.Bar(x=df.index, y=df['MACD'], name='MACD 柱状图', marker_color=m_colors), row=3, col=1)

        theme_title = f"🏰 EMPIRE QUANT v3.5.0 - EXECUTION SIGN: [{operation_type}] FOR {name}"
        fig.update_layout(
            title=dict(text=theme_title, x=0.01, font=dict(color='#ffffff' if operation_type=="BUY" else '#ff4d4d', size=16)),
            template="plotly_dark", paper_bgcolor='#0e0e0e', plot_bgcolor='#141414',
            xaxis3_rangeslider_visible=False, height=650, margin=dict(l=40, r=30, t=80, b=40)
        )
        fig.update_yaxes(gridcolor='#222222', zerolinecolor='#333333')
        fig.update_xaxes(gridcolor='#222222')
        fig.show()

    # ==========================================
    # 中央滚动扫描控制核心
    # ==========================================
    def execute_loop_inspection(self, target_epochs=2):
        """ 启动多周期滚动控制总线 """
        print("\n" + "🏰 " * 10 + "帝国中央控制总线 v3.5.0 · 滚动点火" + "🏰 " * 10)

        for epoch in range(1, target_epochs + 1):
            print(f"\n{"🔄 " * 5} 正在启动第 [{epoch}/{target_epochs}] 轮全生命周期资产矩阵巡检 {"🔄 " * 5}")

            for code, name in PORTFOLIO_MAP.items():
                print(f"\n📡 [接入链路] 正在拉取标的数据流: [{code} | {name}] ...")
                raw_data = self.fetch_real_market_data(stock_code=code, lookback_days=120)

                if raw_data is None or raw_data.empty:
                    continue

                processed_df = self.calculate_indicators(raw_data)
                latest_metrics = processed_df.iloc[-1]

                current_close = float(latest_metrics['Close'])
                current_bias = float(latest_metrics['Bias20'])
                current_macd = float(latest_metrics['MACD'])
                macd_slope = float(latest_metrics['MACD_Slope'])

                # ----------------------------------------------------
                # 场景一：若该资产已经在持仓中 -> 执行实时持仓防御监控
                # ----------------------------------------------------
                if code in self.positions:
                    pos_info = self.positions[code]
                    print(f"🛡️ [持仓护航中] {name} 持仓价: {pos_info['entry_price']:.2f} | 当前实时价: {current_close:.2f}")

                    # 【核心破位清仓算法】：价格跌破 MA20 趋势保护线，或者 MACD 出现严重动能衰竭
                    exit_condition_trend = current_close < latest_metrics['MA20']
                    exit_condition_macd = current_macd < 0 and macd_slope < 0

                    if exit_condition_trend or exit_condition_macd:
                        revenue = pos_info['shares'] * current_close
                        self.available_cash += revenue
                        print(f"💥 [💥 趋势破位强制清仓] 标的 {name} ({code}) 触发风控红线！已执行清仓交割。")
                        print(f"💰 释出安全现金垫: {revenue:,.2f} RMB。")

                        # 渲染清仓时的可视化图表
                        self.render_interactive_dashboard(processed_df, ticker=code, name=name, operation_type="CRITICAL_SELL")
                        del self.positions[code]  # 从持仓账本中剔除
                    else:
                        print(f"✅ [防护安全] {name} 指标健康，继续锁仓留存。")

                # ----------------------------------------------------
                # 场景二：若该资产不在持仓中 -> 执行共振买入准入机制
                # ----------------------------------------------------
                else:
                    if len(self.positions) >= self.max_positions:
                        print(f"🛡️ [席位饱和] 当前持仓集群已满 ({len(self.positions)}), 拒绝增加新标的准入。")
                        continue

                    # 买入条件：价格 > MA20 且 (MACD > 0 或 红柱向上加速)
                    condition_trend = current_close > latest_metrics['MA20']
                    condition_momentum = current_macd > 0 or macd_slope > 0

                    if condition_trend and condition_momentum:
                        # 触发 [乖离度防追高拦截]
                        if current_bias > self.bias_limit:
                            print(f"🛑 [风控过热拦截] 标的 {name} 偏离 MA20 过远 ({current_bias*100:.1f}%), 拒绝并网接盘。")
                            continue

                        print(f"🟢 [技术初筛通过] 发现标的 {name} 触发右侧双重动能共振。")

                        # 调用 AI 舆情风控审计
                        ai_clearance = self.run_ai_fraud_audit(ticker=code, name=name)
                        if not ai_clearance:
                            print(f"🛑 [AI 一票否决] {name} 舆情或财务链条存在暗雷，强制撤单。")
                            continue

                        # 执行主板资金划转与下单建仓
                        shares_to_buy = int((self.position_limit / current_close) // 100) * 100
                        cost = shares_to_buy * current_close

                        if shares_to_buy > 0 and self.available_cash >= cost:
                            self.available_cash -= cost
                            self.positions[code] = {
                                'name': name, 'shares': shares_to_buy,
                                'entry_price': current_close, 'cost': round(cost, 2)
                            }
                            print(f"💰 [主板资产划转] 成功建立 {name} 持仓席位。划转金额: {cost:,.2f} RMB。")

                            # 渲染建仓看板
                            self.render_interactive_dashboard(processed_df, ticker=code, name=name, operation_type="BUY")
                    else:
                        print(f"🔴 [技术拦截] 标的 {name} 处于弱势震荡波段，资产不予并网。")

                # 🛡️ 强制阻尼延迟：彻底消除海外云环境 IP 轮询特征
                time.sleep(random.uniform(2.0, 3.5))

            # 每轮循环完毕，打印阶段性复盘总览
            self._print_epoch_report(epoch)

        print("\n" + "👑 " * 10 + "多轮滚动智能巡检全部结束" + "👑 " * 10)

    def _print_epoch_report(self, epoch):
        """ 打印每轮资产实时留存与清流账本 """
        print(f"\n{"📊 " * 8} 第 {epoch} 轮巡检后 - 帝国大厦实时持仓状况书 {"📊 " * 8}")

        display_records = []
        allocated_funds = 0

        for code, info in self.positions.items():
            display_records.append({
                'Ticker': code, 'Asset': info['name'], 'Shares': f"{info['shares']:,}",
                'Entry_Price': info['entry_price'], 'Capital_Allocated': info['cost']
            })
            allocated_funds += info['cost']

        if display_records:
            print(pd.DataFrame(display_records).to_string(index=False))
        else:
            print("⚪ 防御策略生效：当前持仓集群无任何股票符合持股条件，100% 空仓现金避险中。")

        print("-" * 75)
        print(f"🏦 账户可用安全现金储备  : {self.available_cash:,.2f} RMB")
        print(f"👑 帝国集群实时防御总净值: {self.available_cash + allocated_funds:,.2f} RMB")
        print("📊 " * 27 + "\n")

if __name__ == "__main__":
    # ⚠️ 请在这里输入真实的 Gemini API 密匙
    YOUR_REAL_API_KEY = "AIzaSy..."

    # 调动控制总线：本金 100 万，最多 5 只持仓，偏离度阈值限制 15%
    commander = EmpireQuantMatrixLoopTerminal(
        initial_cash=1000000.0,
        max_positions=5,
        gemini_key=YOUR_REAL_API_KEY,
        bias_limit=0.15
    )

    # 启动全自动量化控制滚动矩阵（运行 2 轮完整闭环循环监测）
    commander.execute_loop_inspection(target_epochs=2)


🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 帝国中央控制总线 v3.5.0 · 滚动点火🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 

🔄 🔄 🔄 🔄 🔄  正在启动第 [1/2] 轮全生命周期资产矩阵巡检 🔄 🔄 🔄 🔄 🔄 

📡 [接入链路] 正在拉取标的数据流: [000725 | BOE] ...
🛑 [风控过热拦截] 标的 BOE 偏离 MA20 过远 (15.7%), 拒绝并网接盘。

📡 [接入链路] 正在拉取标的数据流: [600157 | YTP] ...
🟢 [技术初筛通过] 发现标的 YTP 触发右侧双重动能共振。
🔍 [AI 审计节点] 正在跨网深度解剖 YTP (600157) 异动舆情脉络...
💰 [主板资产划转] 成功建立 YTP 持仓席位。划转金额: 199,824.00 RMB。



📡 [接入链路] 正在拉取标的数据流: [601991 | DTG] ...
🛑 [风控过热拦截] 标的 DTG 偏离 MA20 过远 (26.9%), 拒绝并网接盘。

📡 [接入链路] 正在拉取标的数据流: [000767 | JDP] ...
🛑 [风控过热拦截] 标的 JDP 偏离 MA20 过远 (24.6%), 拒绝并网接盘。

📡 [接入链路] 正在拉取标的数据流: [000100 | TCL] ...
🟢 [技术初筛通过] 发现标的 TCL 触发右侧双重动能共振。
🔍 [AI 审计节点] 正在跨网深度解剖 TCL (000100) 异动舆情脉络...
💰 [主板资产划转] 成功建立 TCL 持仓席位。划转金额: 199,793.00 RMB。



📊 📊 📊 📊 📊 📊 📊 📊  第 1 轮巡检后 - 帝国大厦实时持仓状况书 📊 📊 📊 📊 📊 📊 📊 📊 
Ticker Asset  Shares  Entry_Price  Capital_Allocated
600157   YTP 108,600         1.84           199824.0
000100   TCL  45,100         4.43           199793.0
---------------------------------------------------------------------------
🏦 账户可用安全现金储备  : 600,383.00 RMB
👑 帝国集群实时防御总净值: 1,000,000.00 RMB
📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 


🔄 🔄 🔄 🔄 🔄  正在启动第 [2/2] 轮全生命周期资产矩阵巡检 🔄 🔄 🔄 🔄 🔄 

📡 [接入链路] 正在拉取标的数据流: [000725 | BOE] ...
🛑 [风控过热拦截] 标的 BOE 偏离 MA20 过远 (15.7%), 拒绝并网接盘。

📡 [接入链路] 正在拉取标的数据流: [600157 | YTP] ...
🛡️ [持仓护航中] YTP 持仓价: 1.84 | 当前实时价: 1.84
✅ [防护安全] YTP 指标健康，继续锁仓留存。

📡 [接入链路] 正在拉取标的数据流: [601991 | DTG] ...
🛑 [风控过热拦截] 标的 DTG 偏离 MA20 过远 (26.9%), 拒绝并网接盘。

📡 [接入链路] 正在拉取标的数据流: [000767 | JDP] ...
🛑 [风控过热拦截] 标的 JDP 偏离 MA20 过远 (24.6%), 拒绝并网接盘。

📡 [接入链路] 正在拉取标的数据流: [000100 | TCL] ...
🛡️ [持仓护航中] TCL 持仓价: 4.43 | 当前实时价: 4.43
✅ [防护安全] TCL 指标健康，继续锁仓留存。

📊 📊 📊 📊 📊 📊 📊 📊  第 2 轮巡检后 - 帝国大厦实时持仓状况书 📊 📊 📊 📊 📊 📊 📊 📊 
Ticker Asset  

In [10]:
import datetime
import time
import random
import warnings
import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import akshare as ak
import google.generativeai as genai

# 全局战略屏蔽：阻断底层冗余警告，保持中央控制台绝对纯净
warnings.filterwarnings('ignore')

# 帝国核心防御资产集群 (长官的5大核心标的)
PORTFOLIO_MAP = {
    "000725": "京东方A",
    "600157": "永泰能源",
    "601991": "大唐发电",
    "000767": "晋控电力",
    "000100": "TCL科技"
}

class EmpireQuantPrecisionTerminal:
    """
    🏰 EMPIRE QUANT LAB - 多资产全生命周期自动化滚动控制总线 (v3.5.5 动态清流版)
    核心机制：
    1. 动能共振准入 + Bias 乖离率防追高拦截
    2. MA20 趋势破位提前清仓 + MACD 衰竭动能一票否决
    3. 全自动多周期(Epoch)时间轴流控演进
    """
    def __init__(self, initial_cash=1000000.0, max_positions=5, gemini_key="", bias_limit=0.15):
        # 资金风控账本
        self.initial_cash = initial_cash
        self.available_cash = initial_cash
        self.max_positions = max_positions
        self.position_limit = initial_cash / max_positions
        self.bias_limit = bias_limit  # 偏离度风控红线（15%）

        # 动态持仓账本：{ 代码: { 资产详情 } }
        self.positions = {}
        self.audit_logs = {}

        # 浏览器指纹集群
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'
        }

        # 调配 Gemini AI 认知内核
        self.gemini_key = gemini_key
        if self.gemini_key and not self.gemini_key.startswith("AIzaSy..."):
            genai.configure(api_key=self.gemini_key)
            self.ai_model = genai.GenerativeModel('gemini-1.5-flash')
        else:
            self.ai_model = None

    def fetch_real_market_data(self, stock_code, lookback_days=120):
        """ 稳固级日线历史行情数据流同步引擎 """
        symbol_prefix = "sh" if stock_code.startswith("60") else "sz"
        full_symbol = f"{symbol_prefix}{stock_code}"

        for attempt in range(3):
            try:
                df_raw = ak.stock_zh_a_daily(symbol=full_symbol, adjust="qfq")
                if df_raw is not None and not df_raw.empty:
                    df = df_raw[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
                    df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
                    df['Date'] = pd.to_datetime(df['Date'])
                    df.set_index('Date', inplace=True)
                    return df.tail(lookback_days)
            except Exception as e:
                if attempt < 2:
                    wait_sec = (attempt + 1) * 3.0
                    print(f"📡 [链路抖动] 标的 {stock_code} 遭遇阻断。执行第 {attempt+1} 次抗封锁重试，休眠 {wait_sec} 秒...")
                    time.sleep(wait_sec)
                else:
                    print(f"⚠️ [总线断开] {stock_code} 技术降级跳过。")
        return None

    def calculate_indicators(self, df):
        """ 数学指标矩阵：均线簇 + Bias 乖离度 + MACD 动能一阶导 """
        df = df.copy()
        df['MA5'] = df['Close'].rolling(window=5).mean()
        df['MA20'] = df['Close'].rolling(window=20).mean()
        df['Bias20'] = (df['Close'] - df['MA20']) / df['MA20']

        exp1 = df['Close'].ewm(span=12, adjust=False).mean()
        exp2 = df['Close'].ewm(span=26, adjust=False).mean()
        df['DIF'] = exp1 - exp2
        df['DEA'] = df['DIF'].ewm(span=9, adjust=False).mean()
        df['MACD'] = 2 * (df['DIF'] - df['DEA'])
        df['MACD_Slope'] = df['MACD'].diff()
        return df

    def run_ai_fraud_audit(self, ticker, name):
        """ AI 认知内核：全网拦截异动舆情反欺诈审计 """
        if not self.ai_model:
            return True # 未配置Key时自动放行

        print(f"🔍 [AI 审计节点] 正在跨网深度解剖 {name} ({ticker}) 异动舆情脉络...")
        api_url = f"https://xueqiu.com/query/v1/symbol/status.json?count=15&comment=0&symbol={ticker}&hl=0&source=all"

        try:
            res = requests.get(api_url, headers=self.headers, timeout=6)
            status_list = res.json().get('list', [])
            combined_sentiment = "".join([s.get('text', '') for s in status_list])

            if not combined_sentiment or len(combined_sentiment.strip()) < 10:
                return True

            prompt = f"Analyze market discussions for short-seller risk in {name} ({ticker}). Data: {combined_sentiment[:1000]}. Return CONCLUSION: PASSED or VETOED with one reason."
            response = self.ai_model.generate_content(prompt)
            result_text = response.text.strip()
            print(f"🧠 [Gemini 审计报告书]: {result_text}")
            return "VETOED" not in result_text
        except Exception:
            return True

    def render_interactive_dashboard(self, df, ticker, name, operation_type="BUY"):
        """ Plotly 赛博朋克高画质三层交互看板渲染 (修复 Candlestick 属性报错) """
        fig = make_subplots(
            rows=3, cols=1, shared_xaxes=True,
            vertical_spacing=0.03, row_heights=[0.55, 0.18, 0.27]
        )

        # 1. 经典 K 线图
        fig.add_trace(go.Candlestick(
            x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'],
            name=f'{name} K线',
            increasing_line_color='#ff4d4d', decreasing_line_color='#2ecc71'  # 采用标准安全属性配置
        ), row=1, col=1)

        fig.add_trace(go.Scatter(x=df.index, y=df['MA5'], name='MA5 短线', line=dict(color='#ff9900', width=1.5)), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['MA20'], name='MA20 生命周期', line=dict(color='#00bcff', width=1.5)), row=1, col=1)

        # 2. 量能图
        v_colors = ['#ff4d4d' if c >= o else '#2ecc71' for c, o in zip(df['Close'], df['Open'])]
        fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='成交量', marker_color=v_colors, opacity=0.85), row=2, col=1)

        # 3. MACD 指标副图
        fig.add_trace(go.Scatter(x=df.index, y=df['DIF'], name='DIF', line=dict(color='#ffffff', width=1.2)), row=3, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['DEA'], name='DEA', line=dict(color='#ffff00', width=1.2)), row=3, col=1)
        m_colors = ['#ff4d4d' if m >= 0 else '#2ecc71' for m in df['MACD']]
        fig.add_trace(go.Bar(x=df.index, y=df['MACD'], name='MACD柱', marker_color=m_colors), row=3, col=1)

        # 样式调教
        title_color = '#ff4d4d' if "SELL" in operation_type else '#00ffcc'
        fig.update_layout(
            title=dict(text=f"🏰 EMPIRE QUANT v3.5.5 - 信号捕捉: [{operation_type}] -> {name}({ticker})", x=0.01, font=dict(color=title_color, size=15)),
            template="plotly_dark", paper_bgcolor='#0e0e0e', plot_bgcolor='#141414',
            xaxis3_rangeslider_visible=False, height=600, margin=dict(l=40, r=30, t=80, b=40)
        )
        fig.show()

    def execute_loop_inspection(self, target_epochs=2):
        """ 启动多周期滚动控制总线 """
        print("\n" + "🏰 " * 8 + "中央量化控制总线 v3.5.5 · 滚动监测启动" + "🏰 " * 8)

        for epoch in range(1, target_epochs + 1):
            print(f"\n🔄 ======= 【第 {epoch} / {target_epochs} 轮全局巡检演进中】 =======")

            for code, name in PORTFOLIO_MAP.items():
                raw_data = self.fetch_real_market_data(stock_code=code, lookback_days=120)
                if raw_data is None or raw_data.empty:
                    continue

                processed_df = self.calculate_indicators(raw_data)
                latest = processed_df.iloc[-1]

                current_close = float(latest['Close'])
                current_bias = float(latest['Bias20'])
                current_macd = float(latest['MACD'])
                macd_slope = float(latest['MACD_Slope'])
                ma20_line = float(latest['MA20'])

                # ----------------------------------------------------
                # 场景一：资产在持仓账本中 -> 动态拦截趋势破位与动能衰竭
                # ----------------------------------------------------
                if code in self.positions:
                    pos_info = self.positions[code]
                    print(f"🛡️ [持仓护航] {name} 建仓价: {pos_info['entry_price']:.2f} | 现价: {current_close:.2f} | MA20防线: {ma20_line:.2f}")

                    # 趋势防线破位算法 (价格跌破 MA20 趋势线，或者 MACD 彻底转入负向死叉衰竭)
                    exit_trend = current_close < ma20_line
                    exit_macd = current_macd < 0 and macd_slope < 0

                    if exit_trend or exit_macd:
                        revenue = pos_info['shares'] * current_close
                        self.available_cash += revenue
                        reason = "跌破 MA20 趋势线" if exit_trend else "MACD 动能彻底衰竭"
                        print(f"💥 [💥 趋势防御平仓触发] {name}({code}) 触发风控红线 ({reason})！强行执行并网清仓。")
                        print(f"💰 释出安全现金垫: {revenue:,.2f} RMB。")

                        self.render_interactive_dashboard(processed_df, ticker=code, name=name, operation_type="TREND_SELL")
                        del self.positions[code]
                    else:
                        print(f"✅ [指标持仓健康] {name} 锁仓留存中。")

                # ----------------------------------------------------
                # 场景二：资产未持仓 -> 动能共振准入校验
                # ----------------------------------------------------
                else:
                    condition_trend = current_close > ma20_line
                    condition_momentum = current_macd > 0 or macd_slope > 0

                    if condition_trend and condition_momentum:
                        # 触发 [乖离度防追高拦截]
                        if current_bias > self.bias_limit:
                            print(f"🛑 [防追高拦截] {name} 偏离 MA20 过远 (+{current_bias*100:.1f}%), 拒绝高位接盘。")
                            continue

                        print(f"🟢 [技术初筛通过] {name} 右侧动能共振成立。")

                        # AI 反欺诈审计
                        if not self.run_ai_fraud_audit(ticker=code, name=name):
                            print(f"🛑 [AI 一票否决] {name} 舆情链条存在尾部黑天鹅风险，撤单拦截。")
                            continue

                        if len(self.positions) >= self.max_positions:
                            print(f"🛡️ [席位饱和] 持仓集群已满 ({len(self.positions)}), 放弃增补新标的。")
                            continue

                        # 计算买入股数并切分头寸
                        shares_to_buy = int((self.position_limit / current_close) // 100) * 100
                        cost = shares_to_buy * current_close

                        if shares_to_buy > 0 and self.available_cash >= cost:
                            self.available_cash -= cost
                            self.positions[code] = {
                                'name': name, 'shares': shares_to_buy,
                                'entry_price': current_close, 'cost': round(cost, 2)
                            }
                            print(f"💰 [主板资产划转] 成功建立 {name} 持仓。划转金额: {cost:,.2f} RMB。")
                            self.render_interactive_dashboard(processed_df, ticker=code, name=name, operation_type="BUY")
                    else:
                        print(f"🔴 [技术拦截] {name} 属于非多头右侧活跃波段，不予并网。")

                time.sleep(random.uniform(1.5, 2.5)) # 防高频请求限流

            self._print_epoch_report(epoch)
        print("\n" + "👑 " * 8 + "帝国全生命周期多周期智能巡检全部演进结束" + "👑 " * 8)

    def _print_epoch_report(self, epoch):
        print(f"\n📊 第 {epoch} 轮巡检后 - 帝国大厦实时资产负债表 📊")
        display_records = []
        allocated_funds = 0
        for code, info in self.positions.items():
            display_records.append({
                '代码': code, '标的名称': info['name'], '持股数量': f"{info['shares']:,}",
                '建仓价': info['entry_price'], '分配头寸': f"{info['cost']:,.2f}"
            })
            allocated_funds += info['cost']

        if display_records:
            print(pd.DataFrame(display_records).to_string(index=False))
        else:
            print("⚪ 防御空仓中：当前没有符合右侧多头持仓条件的股票，全额安全现金避险。")
        print("-" * 65)
        print(f"🏦 账户可用现金储备: {self.available_cash:,.2f} RMB")
        print(f"👑 帝国集群实时总净值: {self.available_cash + allocated_funds:,.2f} RMB\n")

if __name__ == "__main__":
    # 如果要开启 Gemini 舆情风控审计，请输入真实的 API Key
    YOUR_GEMINI_KEY = "AIzaSy..."

    # 初始化控制台：100万总本金，单股限额20万，偏离度上限15%
    commander = EmpireQuantPrecisionTerminal(
        initial_cash=1000000.0,
        max_positions=5,
        gemini_key=YOUR_GEMINI_KEY,
        bias_limit=0.15
    )

    # 模拟运行 2 轮时间演进，观察持仓在发生变化时的“买入 -> 防护 -> 趋势跌破自动平仓”流程
    commander.execute_loop_inspection(target_epochs=2)


🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 中央量化控制总线 v3.5.5 · 滚动监测启动🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 

🔄 ======= 【第 1 / 2 轮全局巡检演进中】 =======
🛑 [防追高拦截] 京东方A 偏离 MA20 过远 (+15.7%), 拒绝高位接盘。
🟢 [技术初筛通过] 永泰能源 右侧动能共振成立。
💰 [主板资产划转] 成功建立 永泰能源 持仓。划转金额: 199,824.00 RMB。


🛑 [防追高拦截] 大唐发电 偏离 MA20 过远 (+26.9%), 拒绝高位接盘。
🛑 [防追高拦截] 晋控电力 偏离 MA20 过远 (+24.6%), 拒绝高位接盘。
🟢 [技术初筛通过] TCL科技 右侧动能共振成立。
💰 [主板资产划转] 成功建立 TCL科技 持仓。划转金额: 199,793.00 RMB。



📊 第 1 轮巡检后 - 帝国大厦实时资产负债表 📊
    代码  标的名称    持股数量  建仓价       分配头寸
600157  永泰能源 108,600 1.84 199,824.00
000100 TCL科技  45,100 4.43 199,793.00
-----------------------------------------------------------------
🏦 账户可用现金储备: 600,383.00 RMB
👑 帝国集群实时总净值: 1,000,000.00 RMB


🔄 ======= 【第 2 / 2 轮全局巡检演进中】 =======
🛑 [防追高拦截] 京东方A 偏离 MA20 过远 (+15.7%), 拒绝高位接盘。
🛡️ [持仓护航] 永泰能源 建仓价: 1.84 | 现价: 1.84 | MA20防线: 1.79
✅ [指标持仓健康] 永泰能源 锁仓留存中。
🛑 [防追高拦截] 大唐发电 偏离 MA20 过远 (+26.9%), 拒绝高位接盘。
🛑 [防追高拦截] 晋控电力 偏离 MA20 过远 (+24.6%), 拒绝高位接盘。
🛡️ [持仓护航] TCL科技 建仓价: 4.43 | 现价: 4.43 | MA20防线: 4.36
✅ [指标持仓健康] TCL科技 锁仓留存中。

📊 第 2 轮巡检后 - 帝国大厦实时资产负债表 📊
    代码  标的名称    持股数量  建仓价       分配头寸
600157  永泰能源 108,600 1.84 199,824.00
000100 TCL科技  45,100 4.43 199,793.00
-----------------------------------------------------------------
🏦 账户可用现金储备: 600,383.00 RMB
👑 帝国集群实时总净值: 1,000,000.00 RMB


👑 👑 👑 👑 👑 👑 👑 👑 帝国全生命周期多周期智能巡检全部演进结束👑 👑 👑 👑 👑 👑 👑 👑 
